# 静息态头皮脑电信号质量评估算法示例

按顺序运行本 Notebook，可完成脑电读取、预处理、逐通道伪差与高低频异常占比、三类频域指标、每个通道柱状图右侧的地形图，软件左侧表格点击后的 KDE 图，以及三组质量雷达图。
算法调用已安装的 `SignalQualityEval 3.0.0` 二进制包，波形、地形图与 KDE 使用包内接口及配套数据。

本 Notebook 需要 Python 3.12，支持 Windows 与 Linux，默认评估整段文件的 19 个指定 EEG 通道。默认读取相对路径 `data/example.edf`，也可通过环境变量 `EEG_DATA_PATH` 指定自己的文件。
交付文件包括本 Notebook、对应平台的 Python 3.12 wheel 和相应依赖包。先在所选 Jupyter 内核的环境中安装 wheel 及依赖，再从本仓库根目录运行，无需提供或访问软件源码。
Windows 使用 Windows wheel，Linux 使用 Linux wheel，两种平台的二进制文件不能互换。标准分布 CSV、字体等资源随 wheel 安装，不需要另行复制 `Standard` 目录。
依赖包括 `eegaux`、`eegext`、`mne`、`mne_icalabel`、`numpy`、`pandas`、`scipy`、`matplotlib`、`ipykernel`。即使不运行 ICA，当前算法模块仍依赖 `mne_icalabel`。完整依赖版本及安装步骤见本目录的 requirements 文件和 README。

## 0. 环境与参数

### 0.1 确认已安装的二进制包
选择已经安装对应平台 wheel 的 Python 3.12 内核。下面通过 `importlib.metadata` 检查安装版本为 `3.0.0`，并确认实际导入路径与安装记录一致。
随后检查四个核心模块由 Python 扩展加载器加载，即 Windows 的 `.pyd` 或 Linux 的 `.so`。若误加载源码、旧版本或其他环境，会明确停止。
本例不搜索源码目录，不修改 `sys.path`，也不使用源码路径环境变量。

In [ ]:
from pathlib import Path
from importlib import metadata, util
from importlib.machinery import ExtensionFileLoader
import os
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请使用已安装本平台 wheel 的 Python 3.12 内核")

distribution = metadata.distribution("SignalQualityEval")
if distribution.version != "3.0.0":
    raise RuntimeError("请在当前内核安装 SignalQualityEval 3.0.0 wheel")

import SignalQualityEval

package_dir = Path(SignalQualityEval.__file__).resolve().parent
installed_dir = Path(distribution.locate_file("SignalQualityEval")).resolve()
if package_dir != installed_dir or SignalQualityEval.__version__ != "3.0.0":
    raise RuntimeError("导入位置与安装记录不一致，请从独立 notebooks 目录启动并重启内核")

module_locations = {}
for name in [
    "SignalQualityEval.tools.powersg",
    "SignalQualityEval.tools.utils",
    "SignalQualityEval.viz.plot_fun",
    "SignalQualityEval.viz.topomap_compat",
]:
    spec = util.find_spec(name)
    if spec is None or not isinstance(spec.loader, ExtensionFileLoader):
        raise RuntimeError(f"{name} 不是已编译模块，请安装完整 wheel 后重启内核")
    module_locations[name] = Path(spec.origin)

print("版本：", distribution.version)
print("软件入口：", f"{package_dir.name}/{Path(SignalQualityEval.__file__).name}")
print("二进制模块：", [path.name for path in module_locations.values()])

### 0.1.1 导入算法与绘图接口
从已确认的二进制包导入预处理、异常检测、频带指标和绘图函数。下面各节演示这些公开调用方法。
`np.trapz` 在新版 NumPy 中已移除，兼容分支将旧函数名指向同义的 `np.trapezoid`，仅影响本次内核，不修改安装包文件。
`format_notebook_warning` 只保留警告类别和消息，避免将内核脚本的绝对路径写入运行记录。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import warnings
import eegaux.io

from IPython import get_ipython
from IPython.display import display
from matplotlib import font_manager
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.spatial import QhullError
from eegaux.interval import Intervals
from eegaux.preprocessing import set_RFN, set_standard 
from SignalQualityEval.viz.topomap_compat import TopomapMaker, plot_topomap
from SignalQualityEval.tools.utils import (
    preprocess_raw, common_average, get_bad_channels,
    get_Artifact, get_abnormal_inertvals, ica_detect,
)
from SignalQualityEval.tools.powersg import (
    filter_bands, cal_psd_freq, bands_repower, bands_amp,
    detect_bad_channels_psd, get_power_quantile,
)
from SignalQualityEval.viz.plot_fun import plot_segment_raw

if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

mne.set_log_level("ERROR")
print("MNE", mne.__version__, "NumPy", np.__version__)


def format_notebook_warning(message, category, filename, lineno, line=None):
    """输出警告类别与消息，不展示运行机器的脚本路径。"""
    return f"{category.__name__}: {message}\n"


warnings.formatwarning = format_notebook_warning

### 0.1.2 设置内嵌绘图和中文字体
使用 Matplotlib 的 inline 后端，在 Notebook 内显示波形、柱状图、地形图和 KDE 图。
优先使用系统中可用的中文字体，Windows 与 Linux 使用同一段选择代码。只有系统没有中文字体时，才加载随软件交付的 `viz/SimSun.ttc`。
两端字体字形可能略有不同，但统计值、图形布局和色标计算使用相同代码。

In [ ]:
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

font_candidates = [
    "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC",
    "WenQuanYi Micro Hei", "Noto Sans CJK JP", "SimSun",
]
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next((name for name in font_candidates if name in installed_fonts), None)

if chinese_font is None:
    bundled_font = Path(SignalQualityEval.__file__).resolve().parent / "viz" / "SimSun.ttc"
    if not bundled_font.is_file():
        raise FileNotFoundError("缺少中文字体，请安装 Noto Sans CJK 或提供 viz/SimSun.ttc")
    font_manager.fontManager.addfont(str(bundled_font))
    chinese_font = font_manager.FontProperties(fname=str(bundled_font)).get_name()

plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 100
print("绘图字体：", chinese_font)

### 0.2 设置数据文件、评估范围与绘图参数
Windows 与 Linux 共用默认相对路径 `data/example.edf`，以启动 Notebook 的目录为基准。`EEG_DATA_PATH` 可在启动 Jupyter 前指定实际文件，也可修改 `data_path`。Linux 的路径和包名区分大小写。
对外交付时保留通用路径，实际目录放在运行环境配置中。

- `pick_start`、`pick_stop`：相对原文件起点的评估边界，单位为秒。当前 `pick_stop=None` 表示评估到文件末尾，快速试运行可改为 `120.0`。
- `pick_channels`：当前明确选取 19 个头皮 EEG 通道，名称对应下一节标准化后的大写名称。设为 `None` 时选择标准化后的全部 EEG 通道。
- `pick_times`：相对裁剪后 Raw 起点的波形显示窗口。`(10, 20)` 对应原文件的 `pick_start + 10` 至 `pick_start + 20` 秒，只改变绘图范围，不改变统计范围。
- `resample`、`target_band`：目标采样率和预处理带通范围。实际采样率取目标值与原采样率的较小值，默认 256 Hz、1–70 Hz。后续频域指标固定分析 1–40 Hz，修改预处理频带时需考虑与这些指标及标准分布的可比性。
- `abnormal_win`：高低频异常检测的窗口长度，默认 1 秒，步长为窗口的一半。它不控制第 3 节振幅伪差检测器内部固定的 0.1 秒窗口。
- `y_scale`：波形幅值标尺，单位为 µV。处理前后均使用同一通道顺序、显示时间和 75 µV 标尺，便于比较。

In [ ]:
data_path = Path(os.environ.get("EEG_DATA_PATH", "data/example.edf")).expanduser()

pick_start = 0.0
pick_stop = None
pick_channels = ['FP1', 'FP2', 'FZ', 'F3', 'F4', 'F7', 'F8', 'CZ', 'C3', 'C4', 'PZ', 'P3', 'P4', 'T3', 'T4', 'T5', 'T6', 'O1', 'O2']

pick_times = (100.0, 110.0)
y_scale = 75
resample = 256
target_band = (1, 70)
abnormal_win = 1.0

## 1. 脑电读取与原始波形

### 1.1 使用 `eegaux.io.load_data` 读取并标准化通道
调用方式为 `raw, data_type = eegaux.io.load_data(str(data_path), preload=False)`。
第一个返回值为 MNE Raw，第二个返回值为读取器的辅助类型信息。`preload=False` 先读取文件信息，下一步只载入选定片段。内存足够时也可使用 `preload=True`。

`set_standard(raw, kind="standard_1020", ch_upper=True)` 原地匹配标准 10–20 电极名称、设置标准电极位置，并统一名称为大写。无法匹配标准位置的通道仍会保留，不能据此假定所有通道都有有效坐标。下方分别打印标准化前后的名称，后续 `pick_channels` 使用标准化后的名称。若采集布局不同，应选择相符的标准或提供实际电极坐标。
标准化仅整理通道和位置信息，此时尚未进行带通滤波、平均参考或 ICA。电极坐标也供后续可选的 ICA 分类使用。
读取前检查文件是否存在，未配置数据时会提示设置 `data_path` 或 `EEG_DATA_PATH`。

In [ ]:
if not data_path.is_file():
    raise FileNotFoundError(
        "未找到数据文件，请设置 EEG_DATA_PATH 或将数据放入 data/example.edf"
    )

raw_loaded, data_type = eegaux.io.load_data(str(data_path), preload=False)

print("原始通道：", raw_loaded.ch_names)
print("采样率：", raw_loaded.info["sfreq"], "Hz")
print("总时长：", raw_loaded.n_times / raw_loaded.info["sfreq"], "s")

set_standard(raw_loaded, kind="standard_1020", ch_upper=True)
print("标准化后通道：", raw_loaded.ch_names)
# 如果对于非头皮脑电类通道可以有：
montage = mne.channels.make_standard_montage(kind="standard_1005")
standard_chnames = [ch.upper() for ch in montage.ch_names]
eeg_channels = [ch for ch in raw_loaded.ch_names if ch in standard_chnames]  
print("属于头皮电极的脑电通道记录："  , eeg_channels) 
# 选择属于脑电范畴的记录：
raw_loaded.pick(eeg_channels) 
print("选择属于脑电范畴后的脑电通道：", raw_loaded.ch_names) 



### 1.2 裁剪评估片段并选择通道
与软件顺序一致，先 `crop` 再预处理。MNE 的 `tmax` 包含末端采样点，后续占比分母统一使用 `n_times / sfreq`。
保留 `raw_original` 作为预处理前对照。裁剪后所有算法区间与波形横轴都从 0 秒计时。

In [ ]:
stop = raw_loaded.times[-1] if pick_stop is None else float(pick_stop)
if not 0 <= pick_start < stop <= raw_loaded.times[-1]:
    raise ValueError("请设置位于文件时间范围内的 pick_start 和 pick_stop")

if pick_channels is None:
    pick_channels = raw_loaded.copy().pick("eeg").ch_names
if len(pick_channels) < 2:
    raise ValueError("本例的平均参考与通道间比较至少需要两个 EEG 通道")

raw_original = raw_loaded.copy().crop(tmin=pick_start, tmax=stop)
raw_original.pick(pick_channels).load_data()

if not 0 <= pick_times[0] < pick_times[1] <= raw_original.times[-1]:
    raise ValueError("pick_times 必须位于裁剪后的 Raw 时间范围内")

fig_size = (16, max(5, 0.35 * len(pick_channels)))
print("原文件评估范围：", pick_start, "到", stop, "s")
print("显示的原文件时间：", tuple(pick_start + t for t in pick_times))

### 1.3 使用软件内部方法画原始波形
`plot_segment_raw(raw, start, stop, ...)` 返回 Matplotlib Axes。
内部自动把 V 转成 μV，并保持软件的负向朝上显示方式。`y_scale` 控制通道间距与左侧标尺。

In [ ]:
ax = plot_segment_raw(
    raw_original, start=pick_times[0], stop=pick_times[1],
    fig_size=fig_size, y_scale=y_scale, title="预处理前",
)
ax.set_xlabel("裁剪后时间（s）")
plt.show()


## 2. 预处理与可选的 ICA 去伪差

### 2.1 调用 `preprocess_raw`，按开关选择是否执行 `ica_detect`
`preprocess_raw(raw, pick_channels, resample, l_freq, h_freq, kind_flag)` 完成通道选择、重采样、带通滤波和平均参考，返回处理后的 Raw，同时修改传入对象，因此这里传入 `raw_original.copy()`。
`kind_flag=False` 沿用第 1 节标准化后的通道及电极坐标。当前参数为不高于原采样率的 256 Hz、1–70 Hz 带通，修改入口在第 0.2 节。

**ICA 默认开启**，当前设置为 `use_ica=True`。程序执行 `ica_detect`，并将重建信号用于后续波形、伪差占比和频域统计。设置 `use_ica=False` 可关闭此步骤。

| 参数或返回值 | 含义与调用方式 |
| --- | --- |
| `ica_n_components=5` | 拟合 5 个独立成分。成分数应适合通道数及有效数据秩，平均参考会降低数据秩。 |
| `ica_ex_label` | 指定允许剔除的类别。保留原设置 `eye blink`、`heart beat`、`muscle artifact`，分别对应眼部、心搏和肌肉伪差。只处理眼部伪差时改为 `["eye blink"]`。 |
| `return_exclude_raw=True` | 对全部成分进行 ICLabel 分类，只剔除标签属于指定列表且分类置信度不低于 0.8 的成分，返回重建 Raw。设为 `False` 仍会分类，但返回未剔除成分的输入 Raw。 |
| `ica_record` | 类别到分类置信度的简要字典，并非逐成分结果。同一类别有多个成分时，当前接口只保留最后一个，不能用它统计剔除数量或眼动时间占比。 |
| `raw_processed` | 开关关闭时为常规预处理结果，开启时为 ICA 重建结果。原始 `raw_original` 保留，供波形对照。 |

ICA 分类需要已加载的 EEG 数据、有效电极坐标，以及可用的 ICLabel 推理后端。第 1 节已用 `set_standard` 设置坐标。
当前封装使用 FastICA，输入为 1–70 Hz、共同平均参考后的信号，按指定类别与 0.8 置信度阈值剔除成分。

In [ ]:
target_fs = min(resample, raw_original.info["sfreq"])
if not 0 < target_band[0] < target_band[1] < target_fs / 2:
    raise ValueError("带通范围需大于 0，且上限低于处理后采样率的一半")

raw_processed = preprocess_raw(
    raw_original.copy(), pick_channels=pick_channels,
    resample=target_fs, l_freq=target_band[0], h_freq=target_band[1],
    kind_flag=False,
)

use_ica = True
ica_n_components = 5
ica_ex_label = ["eye blink", "heart beat", "muscle artifact"]
ica_record = None

if use_ica:
    ica_record, raw_processed = ica_detect(
        raw_processed,
        n_components=ica_n_components,
        return_exclude_raw=True,
        ica_ex_label=ica_ex_label,
    )
    display(pd.Series(ica_record, name="类别置信度摘要").to_frame())

print("ICA：", "已执行自动分类与条件剔除" if use_ica else "关闭")

ch_names = raw_processed.ch_names
fs = raw_processed.info["sfreq"]
duration = raw_processed.n_times / fs
print("处理后：", len(ch_names), "个通道", fs, "Hz", duration, "s")

raw_processed.plot_psd(fmin=target_band[0], fmax=target_band[1])

### 2.2 使用相同绘图函数展示处理后的波形
再次调用 `plot_segment_raw`，保持 `pick_times`、`y_scale` 和图幅尺寸与原始波形一致。
`use_ica=False` 时展示常规预处理结果，`True` 时展示 ICA 返回的结果。修改开关后应从第 2 节开始重新运行所有后续单元格，保证图表与统计使用同一份信号。

In [ ]:
ax = plot_segment_raw(
    raw_processed, start=pick_times[0], stop=pick_times[1],
    fig_size=fig_size, y_scale=y_scale, title="预处理后",
)
ax.set_xlabel("裁剪后时间（s）")
plt.show()

## 3. 每个通道的伪差占比

### 3.1 按软件参数识别并排除坏道
`get_bad_channels(raw, start=0, thresh=4.5, mode="both")` 使用线长指标识别坏道，返回通道名数组。
保留全通道的 `raw_processed` 用于展示和频域统计，伪差及高低频异常只在 `raw_detect` 的剩余通道上检测。
坏道在占比表中显示 `NaN` 和“未评估”，而不是当作没有伪差。
频带异常算法需要通道间功率差异，本例要求至少保留三个检测通道，避免两通道平均参考后功率相同导致统计失效。

In [ ]:
bad_channels = get_bad_channels(raw_processed, start=0, thresh=4.5, mode="both")
detect_channels = [ch for ch in ch_names if ch not in bad_channels]
if len(detect_channels) < 3:
    raise ValueError("频带异常检测至少需要三个通道，请检查片段与通道选择")

raw_detect = raw_processed.copy().pick(detect_channels)
channel_status = pd.Series(
    ["未评估（预先排除的坏道）" if ch in bad_channels else "已评估" for ch in ch_names],
    index=ch_names, name="检测状态",
)
print("预先排除的坏道：", bad_channels.tolist())

### 3.2 调用 `get_Artifact` 获取全局及逐通道伪差区间
返回 `(artifact_intervals, artifact_ch_intervals)`，前者是全局区间，后者为 `{通道名: [[起点, 终点], ...]}`，时间均相对于裁剪后的 Raw。
调用保留 `fit_SE` 中的 `win_len=1`、`min_duration=0.5` 写法，但当前实现**未使用这两个参数**，实际固定调用 `ArtifactDetector(min_duration=0, win_len=0.1, amplitude=150.0)`。不要通过这两个参数调节当前版本的检测窗口或最短时长。

In [ ]:
artifact_intervals, artifact_ch_intervals = get_Artifact(
    raw_detect, on_ch=detect_channels, win_len=1, min_duration=0.5, 
)

### 3.3 定义区间统计函数
`interval_ratio(intervals, duration)` 将区间裁切到有效评估范围，合并重叠区间，返回异常时长除以总时长的 0–1 比例。
`channel_ratios(intervals_by_ch, ch_names, duration)` 返回逐通道 Series，字典缺失的通道保留 `NaN`。
这里的分母是整个评估 Raw 的 `n_times / sfreq`，不是波形展示窗口长度，也不是去伪差后的时长。

In [ ]:
def interval_ratio(intervals, duration):
    """计算区间并集占总时长的比例，自动裁切超出范围的部分。"""
    clipped = [
        [max(0, start), min(duration, stop)]
        for start, stop in intervals
        if min(duration, stop) > max(0, start)
    ]
    merged = Intervals.merge(clipped, max_tol_gap=0) if clipped else []
    return sum(stop - start for start, stop in merged) / duration


def channel_ratios(intervals_by_ch, ch_names, duration):
    """按指定通道顺序返回占比，未参与检测的通道为 NaN。"""
    return pd.Series({
        ch: interval_ratio(intervals_by_ch[ch], duration)
        if ch in intervals_by_ch else np.nan
        for ch in ch_names
    })

### 3.4 调用 `eegaux.viz.topomap` 画地形图
`TopomapMaker().get_pos(ch_names)` 按通道顺序返回二维坐标，名称不区分大小写。
同一模块的 `plot_topomap(values, pos, axes=ax, vmin=low, vmax=high, ...)` 可在指定 Axes 上绘图，返回值的第一项是图像，可用于 colorbar。

`TopomapMaker.plot` 本身会调整整幅 Figure 的尺寸、边距，并返回灰度数组。因此组合柱状图与彩色地形图时，直接调用它内部使用的 `plot_topomap`，仍使用该文件的坐标和插值实现。

下面的 `plot_channel_topomap(values, channels, ax, norm, cmap)` 接收一组通道数值、对应名称、目标 Axes 和统一色标。
坐标表仅包含 19 个标准头皮通道，不含 A1、A2。没有坐标或数值为 NaN 的通道不参与插值，原表和柱状图仍保留这些通道。
有效坐标不足三个、重复或共线时，右侧说明无法绘图，不把缺失值补成零。地形图是空间插值，用于显示空间分布，不代表新增测量值。

导入通过随软件交付的 `SignalQualityEval.viz.topomap_compat` 完成，它只适配新版 MNE 的私有接口路径。实际调用仍是 `eegaux.viz.topomap` 与 `_topomap` 的原始坐标和绘图实现，不替换地形图算法。

In [ ]:
topomap_maker = TopomapMaker()


def plot_channel_topomap(values, channels, ax, norm, cmap):
    """使用 eegaux 的坐标与绘图函数，在指定 Axes 画有效通道的地形图。"""
    values = np.asarray(values, dtype=float)
    has_position = np.array([ch.upper() in topomap_maker.POS_DICT for ch in channels])
    valid = has_position & np.isfinite(values)
    names = [ch for ch, keep in zip(channels, valid) if keep]
    ax.set_title(f"地形分布（{len(names)} 个有效通道）")

    if len(names) < 3:
        ax.text(0.5, 0.5, "有效坐标不足三个，无法插值", ha="center", transform=ax.transAxes)
        ax.set_axis_off()
        return None

    pos = topomap_maker.get_pos(names)
    if len(np.unique(pos, axis=0)) != len(pos) or np.linalg.matrix_rank(pos - pos.mean(axis=0)) < 2:
        ax.text(0.5, 0.5, "坐标重复或共线，无法插值", ha="center", transform=ax.transAxes)
        ax.set_axis_off()
        return None

    try:
        result = plot_topomap(
            values[valid], pos, axes=ax, vmin=norm.vmin, vmax=norm.vmax,
            cmap=cmap, contours=0, sensors=True, show=False,
        )
    except QhullError:
        ax.text(0.5, 0.5, "有效坐标无法完成空间插值", ha="center", transform=ax.transAxes)
        ax.set_axis_off()
        return None

    image = result[0]
    image.set_norm(norm)
    return image

### 3.4.1 为每个柱状图添加右侧地形图，并统一整组范围
`plot_channel_bars(table, ylabel, title, percent=False)` 输入“行=通道、列=指标”的数值表，每个指标占一行，依次显示柱状图、地形图、colorbar。
同一次调用中的所有柱状图 y 轴和所有地形图 colorbar 使用完全相同的范围与配色。
百分比统一为 0–100%，其他指标统一为 0 到整张表有限最大值的 1.05 倍，全零或全缺失时使用 0–1。
因此同一组频带可以直接比较大小，A1、A2 等无坐标通道也参与全组范围计算。

柱状图保留全部通道，NaN 标为“未评估”。地形图只插值有效头皮通道，插值超出范围时颜色饱和，不自动改变色标。
调用示例见下节的 `plot_channel_bars(artifact_table[["伪差占比（%）"]], ...)`，后面的每组频带沿用同一函数。

In [ ]:
def plot_channel_bars(table, ylabel, title, percent=False):
    """每行绘制通道柱状图和地形图，整张表共享 y 轴与 colorbar 范围。"""
    all_values = table.to_numpy(dtype=float)
    finite = all_values[np.isfinite(all_values)]
    upper = float(finite.max()) * 1.05 if finite.size and finite.max() > 0 else 1.0
    norm = Normalize(vmin=0, vmax=100 if percent else upper, clip=True)
    cmap = plt.get_cmap("jet")

    fig, axes = plt.subplots(
        len(table.columns), 3,
        figsize=(max(16, 0.5 * len(table) + 5), 3.5 * len(table.columns)),
        gridspec_kw={"width_ratios": [4.5, 1.3, 0.09]}, squeeze=False,
    )
    positions = np.arange(len(table))
    channels = table.index.tolist()

    for (ax, topo_ax, colorbar_ax), column in zip(axes, table.columns):
        values = table[column].to_numpy(dtype=float)
        ax.bar(positions, values, color=cmap(norm(values)), width=0.7)
        ax.set_xticks(positions)
        ax.set_xticklabels(channels, rotation=60, ha="right")
        ax.set_title(f"{title} · {column}")
        ax.set_ylabel(ylabel)
        ax.set_xlabel("通道")
        ax.set_ylim(norm.vmin, norm.vmax)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)
        for position in positions[~np.isfinite(values)]:
            ax.text(position, 0.03, "未评估", rotation=90, ha="center",
                    transform=ax.get_xaxis_transform(), fontsize=8)

        image = plot_channel_topomap(values, channels, topo_ax, norm, cmap)
        mappable = image if image is not None else ScalarMappable(norm=norm, cmap=cmap)
        colorbar = fig.colorbar(mappable, cax=colorbar_ax)
        colorbar.set_label(ylabel)
        colorbar.set_ticks(np.linspace(norm.vmin, norm.vmax, 5))

    missing = [ch for ch in channels if ch.upper() not in topomap_maker.POS_DICT]
    note = "地形图仅插值有效头皮通道，NaN 不补零"
    if missing:
        note += "，无坐标通道仅在柱状图中显示：" + "、".join(missing)
    fig.text(0.06, 0.01, note, fontsize=9)
    fig.subplots_adjust(left=0.06, right=0.94, bottom=0.9 / len(table.columns) / 3.5,
                        top=0.92 if len(table.columns) == 1 else 0.97, hspace=0.65, wspace=0.25)
    return fig

### 3.5 输出伪差时长、占比表和柱状图
`artifact_ratio` 保存 0–1 比例，表格与图显示百分比。全局伪差占比由全局区间单独计算，不是通道占比的简单平均。

In [ ]:
artifact_ratio = channel_ratios(artifact_ch_intervals, ch_names, duration)
artifact_table = pd.DataFrame({
    "检测状态": channel_status,
    "伪差时长（s）": artifact_ratio * duration,
    "伪差占比（%）": artifact_ratio * 100,
})

display(artifact_table.round(3))
print(f"全局伪差占比：{interval_ratio(artifact_intervals, duration):.2%}")
fig = plot_channel_bars(artifact_table[["伪差占比（%）"]], "占比（%）", "波幅伪差", percent=True)
plt.show()

### 3.6 定义区间高亮格式并展示伪差波形
`channel_highlights(intervals_by_ch, color, ch_names)` 把通道区间转成软件的 `highlights_h` 格式：`[标签, 颜色, 起点, 终点, 通道序号]`。
红色为逐通道伪差，橙色为预先排除的坏道。展示 `pick_times` 内波形，区间由包内绘图函数裁切到可见范围。

In [ ]:
def channel_highlights(intervals_by_ch, color, ch_names):
    """把秒区间字典转换成 plot_segment_raw 的逐通道高亮列表。"""
    return [
        [None, color, start, stop, ch_names.index(ch)]
        for ch, intervals in intervals_by_ch.items()
        for start, stop in intervals
    ]


bad_highlights = [[None, "orange", 0, duration, ch_names.index(ch)] for ch in bad_channels]
artifact_highlights = channel_highlights(artifact_ch_intervals, "red", ch_names)

ax = plot_segment_raw(
    raw_processed, start=pick_times[0], stop=pick_times[1],
    highlights_h=artifact_highlights + bad_highlights,
    fig_size=fig_size, y_scale=y_scale, title="伪差（红）与预先排除的坏道（橙）",
)
ax.set_xlabel("裁剪后时间（s）")
plt.show()

## 4. 低频、高频异常占比与波形

### 4.1 封装软件的频带异常调用顺序
`detect_band_intervals(raw, band, win, thresh)` 返回逐通道异常区间字典。
按 `Analyze_nosie_patten.abnormal_intervals` 的顺序，先 `set_RFN` 滤波，再平均参考，最后调用原函数 `get_abnormal_inertvals`（沿用接口拼写）。
`set_RFN` 原地修改 Raw，不返回 Raw。窗口步长为 `win / 2`，`scale=1e6` 将 V 转为 μV，`max_tol_gap=0` 只合并相接或重叠区间。

In [ ]:
def detect_band_intervals(raw, band, win, thresh):
    """复用软件滤波、平均参考和频带异常检测，返回局部秒区间。"""
    filtered = raw.copy()
    set_RFN(filtered, None, band[0], band[1])
    filtered.apply_function(common_average, channel_wise=False)

    return get_abnormal_inertvals(
        filtered.get_data(), filtered.info["sfreq"], win=win,
        scale=1e6, ch_names=filtered.ch_names, max_tol_gap=0, thresh=thresh,
    )

### 4.2 运行低频和高频异常检测
沿用 GUI 参数：低频 1–13 Hz、`thresh=3.5`，高频 13–40 Hz、`thresh=6`，默认 `win=1` 秒。
这里的占比表示异常区间的时间占比，不是该频带的能量占比。两种异常区间可能重叠，不能直接相加当作总异常比例。

In [ ]:
low_intervals = detect_band_intervals(raw_detect, band=(1, 13), win=abnormal_win, thresh=3.5)
high_intervals = detect_band_intervals(raw_detect, band=(13, 40), win=abnormal_win, thresh=6)

abnormal_table = pd.DataFrame({
    "低频异常（%）": channel_ratios(low_intervals, ch_names, duration) * 100,
    "高频异常（%）": channel_ratios(high_intervals, ch_names, duration) * 100,
})
display(pd.concat([channel_status, abnormal_table.round(3)], axis=1))

fig = plot_channel_bars(abnormal_table, "占比（%）", "频带异常", percent=True)
plt.show()

### 4.3 展示 `pick_times` 内的异常区间
保持相同波形和标尺，用浅蓝表示低频异常，紫色表示高频异常，橙色表示预先排除的坏道。
这里显示预处理后 1–70 Hz 波形上的异常标记，与软件主波形的展示方式相同，不把分频滤波结果替换为主波形。

In [ ]:
low_highlights = channel_highlights(low_intervals, "deepskyblue", ch_names)
high_highlights = channel_highlights(high_intervals, "mediumpurple", ch_names)

ax = plot_segment_raw(
    raw_processed, start=pick_times[0], stop=pick_times[1],
    highlights_h=low_highlights + high_highlights + bad_highlights,
    fig_size=fig_size, y_scale=y_scale,
    title="低频异常（浅蓝）、高频异常（紫）与坏道（橙）",
)
ax.set_xlabel("裁剪后时间（s）")
plt.show()

## 5. 相对频带功率 `bands_repower`

### 5.1 按软件方式准备干净片段
复用 `Intervals.difference` 去除全局伪差，只保留**严格超过 10 秒**的连续片段，再拼接为通道 × 采样点矩阵。
与软件 `sgs_power_abnormal` 一致，频域统计保留预处理后的全部所选通道，数据转为 μV。全局伪差来自排除坏道后的检测，因此“干净”仅指通过该全局伪差筛选，不代表每个通道均无异常。
通道标签始终取自同一个 Raw，确保与矩阵行一致。拼接边界也可能影响滤波与 PSD，本例保留软件已有处理方式。
若没有满足条件的片段，明确停止频域部分，不使用未筛选数据替代。

In [ ]:
clean_intervals = Intervals.difference([[0, duration]], artifact_intervals)
clean_intervals = [interval for interval in clean_intervals if interval[1] - interval[0] > 10]
if not clean_intervals:
    raise ValueError("没有超过 10 秒的干净片段，请调整评估时间范围")

power_data = raw_processed.get_data()
data_uv = np.concatenate([
    power_data[:, int(start * fs):int(stop * fs)]
    for start, stop in clean_intervals
], axis=-1) * 1e6
power_channels = raw_processed.ch_names

display(pd.DataFrame(clean_intervals, columns=["干净片段起点（s）", "干净片段终点（s）"]))
print("用于频域统计的总时长：", data_uv.shape[1] / fs, "s")

### 5.2 调用 `cal_psd_freq` 计算 PSD
`cal_psd_freq(data, fs, fmin=1, fmax=40)` 返回 `(psd, freqs)`。输入为 μV 时 PSD 单位为 μV²/Hz。
软件使用 Welch 法，FFT 窗长 2 秒、重叠 1 秒。后面的对数 PSD 距离要求 PSD 为正且有限。

In [ ]:
if data_uv.shape[1] < int(2 * fs):
    raise ValueError("PSD 至少需要 2 秒数据")

psd, freqs = cal_psd_freq(data_uv, fs, fmin=1, fmax=40)
if not np.all(np.isfinite(psd)) or np.any(psd <= 0):
    raise ValueError("PSD 含零或非有限值，请检查平直通道与输入数据")

print("PSD 形状：", psd.shape)
print("频率范围：", freqs[0], "到", freqs[-1], "Hz")

### 5.3 定义频带结果整理函数
`feature_table(result, prefix, channels, bands)` 把算法返回的一行宽表转成“行=通道、列=频带”的表格。
通过完整列名读取，不拆分通道名，因此也支持通道名称自身含下划线的情况。

In [ ]:
def feature_table(result, prefix, channels, bands):
    """将 bpre、bpamp 或 psdist 的宽表转成通道 × 频带表。"""
    return pd.DataFrame({
        band: [result.at[0, f"{prefix}_{ch}_{band}"] for ch in channels]
        for band in bands
    }, index=pd.Index(channels, name="通道"))

### 5.4 调用 `bands_repower` 相对频带功率并展示每通道结果
`bands_repower(psd, freqs, ch_names)` 返回列名为 `bpre_通道_频带` 的一行 DataFrame。
计算各频带积分功率除以 1–40 Hz 积分功率，原始值为 0–1 比例，展示时乘 100。
固定频带来自 `filter_bands`：delta 1–4、theta 4–8、alpha 8–13、beta 13–30、gamma 30–40 Hz。

In [ ]:
repower_result = bands_repower(psd, freqs, power_channels)
repower_table = feature_table(repower_result, "bpre", power_channels, filter_bands)

display((repower_table * 100).round(3).rename_axis(columns="频带相对功率（%）"))
fig = plot_channel_bars(repower_table * 100, "相对功率（%）", "相对频带功率", percent=True)
plt.show()

## 6. 频带平均绝对幅值 `bands_amp`

`bands_amp(data, fs, ch_names)` 对每个频带做 FIR 滤波，再求每通道 `mean(abs(x))`，返回列名为 `bpamp_通道_频带` 的一行 DataFrame。
这里输入 `data_uv`，所以结果单位为 μV。该指标不是 RMS，也不是时间占比。

In [ ]:
amp_result = bands_amp(data_uv, fs, power_channels)
amp_table = feature_table(amp_result, "bpamp", power_channels, filter_bands)

display(amp_table.round(3).rename_axis(columns="频带平均绝对幅值（μV）"))
fig = plot_channel_bars(amp_table, "平均绝对幅值（μV）", "频带波幅均值")
plt.show()

## 7. PSD 通道距离与异常导联 `detect_bad_channels_psd`

### 7.1 调用检测函数并展示各频带距离
`detect_bad_channels_psd(psd, freqs, ch_names, threshold=3, detect_band="full")` 返回 `(psd_bad_channels, psdist_result)`。
函数默认 `threshold=10`，这里显式使用软件统一入口的 `3`。以 `full` 的 Q1/Q3 ± 3 × IQR 判定异常导联，`full` 表示 1–40 Hz 频带。
各频带距离是 `log10(PSD)` 的通道两两绝对差，先对频率平均，再对所有通道平均（包含自身的零距离）。它是无量纲距离，不是 dB，也不是异常时间占比。该接口的距离计算使用频带左闭右开边界。

In [ ]:
psd_bad_channels, psdist_result = detect_bad_channels_psd(
    psd, freqs, power_channels, threshold=3, detect_band="full",
)
psdist_table = feature_table(psdist_result, "psdist", power_channels, [*filter_bands, "full"])

psd_table = psdist_table.copy()
psd_table["PSD 异常导联"] = psd_table.index.isin(psd_bad_channels)
display(psd_table.round(4))
print("PSD 判定的异常导联：", psd_bad_channels)

fig = plot_channel_bars(psdist_table, "log10 PSD 平均距离", "PSD 通道距离")
plt.show()

### 7.2 `get_power_quantile` 的统一调用方法
`get_power_quantile(data_uv, fs, ch_names)` 在内部依次执行 PSD、相对功率、幅值和 PSD 异常通道检测，返回 `(合并宽表, PSD异常导联列表)`。
下面验证统一入口与第 5–7 节分步调用的结果一致。此函数名称虽包含 quantile，返回值实际是这些指标的宽表。

In [ ]:
all_features, combined_bad_channels = get_power_quantile(data_uv, fs, power_channels)
separate_features = pd.concat([repower_result, amp_result, psdist_result], axis=1)

pd.testing.assert_frame_equal(all_features, separate_features)
assert combined_bad_channels == psd_bad_channels
print("统一入口与分步调用一致，特征列数：", all_features.shape[1])
# print("统一入口的特征名称：", all_features.columns.values.tolist())

## 8. 基于psd特征的统计分布曲线图

### 8.1 加载分布曲线和阈值
代码包已经封装基于大样本统计的psd特征统计分布曲线图。
标准文件支持 `bpre`、`bpamp`、`psdist` 与五个频带的 15 种组合，没有 `psdist_full` 的 KDE。

In [ ]:
standard_dir = Path(SignalQualityEval.__file__).resolve().parent / "Standard"
kde_curves = pd.read_csv(standard_dir / "threshold_statisitic_xys.csv")
thresholds = pd.read_csv(standard_dir / "threshold_statisitic.csv").set_index(["feature", "band"])

feature_names = {
    "bpre": "相对频带能量",
    "bpamp": "波幅均值",
    "psdist": "与其他通道差异",
}

### 8.2 数据在大样本分布中的落点
`plot_standard_kde(feature, band, channel, values)` 输入指标名、频带、通道名和第 7.2 节的宽表，返回 Axes。
`channel="total"` 表示当前所选全部通道该频带指标的均值，与软件 total 行的定义一致。
保留软件原图的蓝色分布/填充、绿/橙阈值虚线、红色当前值竖线和圆点以及数值箭头。
`bpre` 必须使用原始 0–1 数值，不传入乘过 100 的展示值。

In [ ]:
def plot_standard_kde(feature, band, channel, values):
    """复现软件左侧频域单元格对应的分布图，支持通道或 total。"""
    key = f"{feature}_{band}"
    if feature not in feature_names or band not in filter_bands:
        raise ValueError("请选择 bpre、bpamp、psdist 和五个标准频带之一")

    if channel == "total":
        value = np.mean([values.at[0, f"{feature}_{ch}_{band}"] for ch in power_channels])
    else:
        value = values.at[0, f"{feature}_{channel}_{band}"]
    if not np.isfinite(value):
        raise ValueError("当前指标值非有限，无法绘制 KDE 标记")

    x_data = kde_curves[f"{key}_xcoords"].dropna().to_numpy()
    y_data = kde_curves[f"{key}_ycoords"].dropna().to_numpy()
    low, high = thresholds.loc[(feature, band), ["thresh_low", "thresh_high"]]

    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    fig.subplots_adjust(left=0.10, right=0.95, top=0.90, bottom=0.12)
    ax.plot(x_data, y_data, color="#3498DB", linewidth=1.8, label="统计分布曲线")
    ax.fill_between(x_data, y_data, alpha=0.15, color="#3498DB")
    ax.axvline(low, color="#2ECC71", linestyle="--", linewidth=1.3, label=f"下阈值: {low:.4f}")
    ax.axvline(high, color="#E67E22", linestyle="--", linewidth=1.3, label=f"上阈值: {high:.4f}")

    y_at_value = y_data[np.argmin(np.abs(x_data - value))]
    ax.axvline(value, color="#E74C3C", linewidth=2, alpha=0.7)
    ax.plot(value, y_at_value, "o", color="#E74C3C", markersize=12,
            zorder=5, markeredgecolor="white", markeredgewidth=1.5)
    ax.annotate(f"{value:.4f}", xy=(value, y_at_value),
                xytext=(15, 15), textcoords="offset points", fontsize=10,
                color="#E74C3C", fontweight="bold",
                arrowprops=dict(arrowstyle="->", color="#E74C3C", lw=1.5))

    ax.set_xlabel("值", fontsize=11)
    ax.set_ylabel("密度", fontsize=11)
    ax.set_title(f"{channel} — {feature_names[feature]} ({band})", fontsize=13)
    ax.legend(fontsize=9, loc="best", framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle="-.")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax

### 8.3 查看单个通道指标大样本分布中的位置
修改 `kde_channel`、`kde_feature`、`kde_band`，即可对应软件左侧表格的另一个单元格。
例如 `"bpre" / "alpha"` 为 alpha 相对功率，`"bpamp" / "beta"` 为 beta 幅值，`"psdist" / "theta"` 为 theta PSD 距离。
将 `kde_channel` 改为 `"total"` 可查看总均值对应图。
当前展示 `O1` 通道的 `alpha` 平均绝对幅值。曲线来自安装包内的标准统计数据，红色标记来自本次评估结果，不是在当前文件上重新拟合 KDE。

In [ ]:
kde_channel = "O1" # power_channels[0]
kde_feature = "bpamp"
kde_band = "alpha"

ax = plot_standard_kde(kde_feature, kde_band, kde_channel, all_features)
plt.show()

### 8.4 同一通道、同一频带的三种 KDE 图
分别调用上述函数展示三个指标，便于与软件同一行的三个频域统计单元格比较。

In [ ]:
for feature in ["bpre", "bpamp", "psdist"]:
    ax = plot_standard_kde(feature, kde_band, kde_channel, all_features)
    plt.show()

## 9. 结果检查与调用位置

### 9.1 检查表格通道、占比和频带结果
保证所有结果表的通道顺序与输入一致，占比在 0–100% 内，三类频域结果均为有限值。
这些检查用于确认调用和结果结构，不把软件的统计阈值解释为临床结论。

In [ ]:
for table in [artifact_table, abnormal_table, repower_table, amp_table, psdist_table]:
    assert table.index.tolist() == ch_names

for ratios in [artifact_ratio, abnormal_table.iloc[:, 0] / 100, abnormal_table.iloc[:, 1] / 100]:
    assert ratios.dropna().between(0, 1).all()

assert np.isfinite(all_features.to_numpy()).all()
assert np.allclose(repower_table.sum(axis=1), 1, atol=1e-6)
print("检查通过：通道对应、占比范围、频域结果与统一调用一致")

### 接口对应关系与使用提醒

| 本例功能 | 调用接口或安装包资源 |
| --- | --- |
| 读取 | `eegaux.io.load_data` |
| 预处理、可选 ICA、伪差、线长坏道、高低频异常 | `SignalQualityEval.tools.utils` |
| 软件的参数与计算顺序 | `SignalQualityEval.fit` 中 `fit_SE` 和 `Analyze_nosie_patten` |
| PSD、相对功率、幅值、PSD 距离 | `SignalQualityEval.tools.powersg` |
| 原始/处理后/异常高亮波形 | `SignalQualityEval.viz.plot_fun` |
| 地形图坐标与空间插值 | `TopomapMaker.get_pos` 和 `plot_topomap` |
| 点击单元格后的 KDE 样式 | `SignalQualityEval.GUI.ResultUI` 的弹窗绘图逻辑 |
| KDE 标准曲线和阈值 | 安装包内 `Standard/threshold_statisitic_xys.csv`、`threshold_statisitic.csv` |

本例先按标准 10–20 布局匹配并统一通道名称为大写，再选取参数区列出的 19 个头皮通道。ICA 默认开启，由第 2 节的 `use_ica` 控制。
本例始终使用数据对应的完整通道名，避免坏道筛选后的标签与频域数据错配。
KDE 图形与软件弹窗使用相同曲线、阈值和标记方法，Notebook 不创建 Qt 弹窗或保存按钮。

当前 `pick_stop=None` 设置为分析整段文件，可通过 `pick_start`、`pick_stop` 缩小评估范围。如要更换波形视窗，只修改 `pick_times` 并重新运行绘图单元格。
重新运行后的检查项应全部通过。接收者只需已安装的二进制包、相应依赖和自己的脑电数据，不需要开发仓库或算法源码。

## 10. 多维质量雷达图

`plot_quality_radars(raw_processed, artifact_ratio, repower_table, amp_table, psdist_table, thresholds, topomap_maker)` 返回三组逐通道归一化指标，并绘制综合异常、相对频带功率偏离度和频带波幅偏离度雷达图。
综合图使用伪差占比、线长偏离度及全频段 PSD 通道差异，另外两图各使用五个标准频带。各维度以 0.5 为阈值参考圈，详细计算及归一化公式见[算法计算方法](算法计算方法.md#19-多维质量雷达图)。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Arc

def plot_quality_radars(
    raw_processed,
    artifact_ratio,
    repower_table,
    amp_table,
    psdist_table,
    thresholds,
    topomap_maker,
    artifact_thresh=0.40,   # 伪差占比阈值设为 40%（映射到雷达图 0.5）
    bad_ch_thresh=4.5,      # 橙色坏道检测默认阈值 4.5（映射到雷达图 0.5）
):
    """
    绘制三组信号质量评估雷达图 + MNE 风格通道空间颜色拓扑图注。
    所有指标经过归一化（0~1）：在阈值范围内 <= 0.5，等于阈值时 = 0.5，超出阈值 > 0.5。
    """
    channels = raw_processed.ch_names
    n_ch = len(channels)
    bands = ["delta", "theta", "alpha", "beta", "gamma"]
    
    # ---------------------------------------------------------
    # 1. 提取电极二维坐标，生成类似 MNE plot_psd 的空间映射颜色
    # ---------------------------------------------------------
    pos_list = []
    valid_mask = []
    for ch in channels:
        if ch.upper() in topomap_maker.POS_DICT:
            pos_list.append(topomap_maker.get_pos([ch])[0])
            valid_mask.append(True)
        else:
            pos_list.append(np.array([0.0, 0.0]))
            valid_mask.append(False)
    pos_arr = np.array(pos_list)
    
    if any(valid_mask):
        center = pos_arr[valid_mask].mean(axis=0)
        rel_pos = pos_arr - center
        angles = np.arctan2(rel_pos[:, 1], rel_pos[:, 0])
        order = np.argsort(angles)
        cmap = plt.get_cmap("turbo")
        ch_colors = np.zeros((n_ch, 4))
        for rank, idx in enumerate(order):
            ch_colors[idx] = cmap(rank / max(1, n_ch - 1))
    else:
        cmap = plt.get_cmap("turbo")
        ch_colors = [cmap(i / max(1, n_ch - 1)) for i in range(n_ch)]

    # ---------------------------------------------------------
    # 2. 定义阈值归一化函数（阈值处 = 0.5，正常 < 0.5，异常 > 0.5）
    # ---------------------------------------------------------
    def norm_upper_only(val, thresh):
        """单侧上界阈值归一化：0 -> 0, thresh -> 0.5, >thresh -> (0.5, 1.0)"""
        val = np.nan_to_num(val, nan=thresh * 2.0)
        val = np.maximum(0.0, val)
        return np.where(
            val <= thresh,
            0.5 * (val / (thresh + 1e-12)),
            1.0 - 0.5 * np.exp(-(val - thresh) / (thresh + 1e-12))
        )

    def norm_two_sided(val, low, high):
        """双侧阈值归一化（对应 Standard 表的 thresh_low 与 thresh_high）：
        区间中心 -> 0, 触及任一阈值边界 -> 0.5, 超出边界 -> (0.5, 1.0)"""
        val = np.nan_to_num(val, nan=high * 1.5)
        mid = 0.5 * (low + high)
        half_w = 0.5 * (high - low) + 1e-12
        dist = np.abs(val - mid)
        return np.where(
            dist <= half_w,
            0.5 * (dist / half_w),
            1.0 - 0.5 * np.exp(-(dist - half_w) / half_w)
        )

    def calc_orange_bad_scores(raw):
        """计算橙色线（get_bad_channels 线长指标）的每通道偏离度评估值（与 thresh=4.5 同尺度）"""
        data = raw.get_data()
        # 逐通道计算一阶差分绝对值均值（即线长 Line Length 能量指标）
        ll = np.mean(np.abs(np.diff(data, axis=-1)), axis=-1)
        med = np.median(ll)
        mad = np.median(np.abs(ll - med))
        scale = 1.4826 * mad if mad > 0 else (np.std(ll) + 1e-12)
        return np.abs(ll - med) / scale

    # ---------------------------------------------------------
    # 3. 计算三个雷达图的归一化数据矩阵 (行=通道, 列=维度)
    # ---------------------------------------------------------
    # 雷达图 1：3个综合维度
    # (1) 伪差片段占比（值域 0~1，直接保留或按 40% 阈值映射到 0.5）
    raw_artifact = artifact_ratio.reindex(channels).to_numpy()
    dim1_artifact = norm_upper_only(raw_artifact, artifact_thresh)
    
    # (2) 能量异常（仅使用橙色坏道检测的线长偏离度评估值，阈值 4.5 映射到 0.5）
    orange_scores = calc_orange_bad_scores(raw_processed)
    dim2_energy = norm_upper_only(orange_scores, bad_ch_thresh)

    # (3) PSD 通道差异异常（采用第7.1节 IQR 阈值 Q3 + 3*IQR 映射到 0.5）
    psdist_full = psdist_table.reindex(channels)["full"].to_numpy()
    q1, q3 = np.nanpercentile(psdist_full, [25, 75])
    iqr = q3 - q1
    psd_iqr_thresh = q3 + 3.0 * iqr if iqr > 0 else np.nanmax(psdist_full) + 1e-6
    dim3_psdist = norm_upper_only(psdist_full, psd_iqr_thresh)

    radar1_data = pd.DataFrame(
        np.column_stack([dim1_artifact, dim2_energy, dim3_psdist]),
        index=channels,
        columns=["伪差片段占比", "能量异常占比", "PSD通道差异"]
    )

    # 雷达图 2：5个频带 repower（基于 Standard 阈值表归一化）
    radar2_dict = {}
    for b in bands:
        low, high = thresholds.loc[("bpre", b), ["thresh_low", "thresh_high"]]
        radar2_dict[b] = norm_two_sided(repower_table.reindex(channels)[b].to_numpy(), low, high)
    radar2_data = pd.DataFrame(radar2_dict, index=channels)

    # 雷达图 3：5个频带 bpamp（基于 Standard 阈值表归一化）
    radar3_dict = {}
    for b in bands:
        low, high = thresholds.loc[("bpamp", b), ["thresh_low", "thresh_high"]]
        radar3_dict[b] = norm_two_sided(amp_table.reindex(channels)[b].to_numpy(), low, high)
    radar3_data = pd.DataFrame(radar3_dict, index=channels)

    # ---------------------------------------------------------
    # 4. 绘制横向排列的 3 个雷达图 + 右侧 Topo 彩色圆点图注
    # ---------------------------------------------------------
    fig = plt.figure(figsize=(20, 5.8), dpi=100)
    gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1, 0.72], wspace=0.32)

    radar_configs = [
        (gs[0, 0], radar1_data, "综合异常评估雷达图"),
        (gs[0, 1], radar2_data, "相对频带功率 (repower) 偏离度"),
        (gs[0, 2], radar3_data, "频带波幅 (bpamp) 偏离度"),
    ]

    for spec, df_radar, title in radar_configs:
        ax = fig.add_subplot(spec, projection="polar")
        labels = df_radar.columns.tolist()
        num_vars = len(labels)
        angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
        angles += angles[:1]

        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(labels, fontsize=10.5)
        ax.set_ylim(0, 1.0)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(["0.25", "0.5(阈值)", "0.75", "1.0"], fontsize=8, color="dimgray")
        ax.grid(True, linestyle="--", alpha=0.45)

        # 绘制 0.5 阈值警戒圆环（红色虚线）
        theta_circle = np.linspace(0, 2 * np.pi, 100)
        ax.plot(theta_circle, np.full_like(theta_circle, 0.5), color="#E74C3C", linestyle="--", linewidth=1.6, zorder=4)
        ax.fill_between(theta_circle, 0, 0.5, color="#2ECC71", alpha=0.06)

        # 逐通道绘制雷达曲线
        for idx, ch in enumerate(channels):
            vals = df_radar.loc[ch].tolist()
            vals += vals[:1]
            ax.plot(angles, vals, color=ch_colors[idx], linewidth=1.5, alpha=0.85)
            ax.scatter(angles[:-1], vals[:-1], color=ch_colors[idx], s=18, zorder=5)

        ax.set_title(title, y=1.10, fontsize=13, fontweight="bold")

    # ---------------------------------------------------------
    # 5. 在第 4 列绘制 MNE 风格 Topomap 传感器颜色图注
    # ---------------------------------------------------------
    ax_topo = fig.add_subplot(gs[0, 3])
    ax_topo.set_aspect("equal")
    ax_topo.axis("off")

    if any(valid_mask):
        valid_pos = pos_arr[valid_mask]
        c_mid = 0.5 * (valid_pos.max(axis=0) + valid_pos.min(axis=0))
        max_r = np.linalg.norm(valid_pos - c_mid, axis=1).max()
        norm_pos = (pos_arr - c_mid) / (max_r + 1e-12) * 0.78
    else:
        norm_pos = pos_arr

    head_r = 0.92
    ax_topo.add_patch(Circle((0, 0), head_r, fill=False, edgecolor="black", linewidth=1.8, zorder=2))
    ax_topo.plot([-0.14, 0, 0.14], [head_r - 0.01, head_r + 0.13, head_r - 0.01], color="black", linewidth=1.8, zorder=2)
    ax_topo.add_patch(Arc((-head_r, 0), 0.12, 0.32, angle=0, theta1=90, theta2=270, color="black", linewidth=1.6))
    ax_topo.add_patch(Arc((head_r, 0), 0.12, 0.32, angle=0, theta1=-90, theta2=90, color="black", linewidth=1.6))

    for idx, ch in enumerate(channels):
        if valid_mask[idx]:
            x, y = norm_pos[idx]
            ax_topo.scatter(x, y, color=ch_colors[idx], s=110, edgecolors="black", linewidths=0.8, zorder=5)
            ax_topo.text(x, y - 0.11, ch, fontsize=8.5, ha="center", va="top", fontweight="bold", color="#222222")

    ax_topo.plot([], [], color="#E74C3C", linestyle="--", linewidth=1.6, label="0.5 阈值边界")
    ax_topo.legend(loc="lower center", bbox_to_anchor=(0.5, -0.10), frameon=False, fontsize=9.5)
    ax_topo.set_xlim(-1.2, 1.2)
    ax_topo.set_ylim(-1.25, 1.25)
    ax_topo.set_title("通道颜色空间图注 (Topomap)", y=1.02, fontsize=12, fontweight="bold")

    plt.show()
    return radar1_data, radar2_data, radar3_data

# 调用函数绘图
radar1_df, radar2_df, radar3_df = plot_quality_radars(
    raw_processed=raw_processed,
    artifact_ratio=artifact_ratio,
    repower_table=repower_table,
    amp_table=amp_table,
    psdist_table=psdist_table,
    thresholds=thresholds,
    topomap_maker=topomap_maker,
)